### This script fixes substring matches of superordinate terms into subordinate ones, which tend to be specifications of the more generic term. You do not need to run this script as the edited results are already in ./wmt_json_translations/results/rel

In [5]:
rel_matches = pd.read_csv(
    "./wmt_json_translations/results/rel/tgt_term_gold.csv",
    sep=",",
    encoding="utf-8-sig"
)

# Automatically take all column names
cols = rel_matches.columns.tolist()

# get term matches (list of binary matches) for the target term
rel_matches = pd.read_csv(
    "./wmt_json_translations/results/rel/tgt_term_gold.csv",
    sep=",",
    encoding="utf-8-sig"
)

rel_matches_hyponym = pd.read_csv(
    "./wmt_json_translations/results/rel/tgt_term_hyponym.csv",
    sep=",",
    encoding="utf-8-sig"
)[cols]

rel_matches_cohyponym = pd.read_csv(
    "./wmt_json_translations/results/rel/tgt_term_co-hyponym.csv",
    sep=",",
    encoding="utf-8-sig"
)[cols]

rel_matches_hypernym = pd.read_csv(
    "./wmt_json_translations/results/rel/tgt_term_hypernym.csv",
    sep=",",
    encoding="utf-8-sig"
)[cols]

In [6]:
import pandas as pd

def has_match(x):
    return str(x).strip() not in ["[]", "", "nan"]


# Copies for edited results
rel_matches_edited = rel_matches.copy()
rel_matches_hyponym_edited = rel_matches_hyponym.copy()
rel_matches_cohyponym_edited = rel_matches_cohyponym.copy()
rel_matches_hypernym_edited = rel_matches_hypernym.copy()

# Keep track of every edit
edit_log = []


for col in cols:

    # ---------------------------------------------------------
    # 1. Gold + Hyponym -> Hyponym survives
    #    Cancel GOLD
    # ---------------------------------------------------------
    mask = (
        rel_matches_edited[col].map(has_match)
        & rel_matches_hyponym_edited[col].map(has_match)
    )

    for idx in rel_matches_edited.index[mask]:
        edit_log.append({
            "row": idx,
            "model": col,
            "edited_category": "gold",
            "original_value": rel_matches_edited.at[idx, col],
            "new_value": "[]",
            "reason": "gold + hyponym match -> hyponym survives"
        })

    rel_matches_edited.loc[mask, col] = "[]"


    # ---------------------------------------------------------
    # 2. Gold + Co-hyponym -> Gold survives
    #    Cancel CO-HYPONYM
    # ---------------------------------------------------------
    mask = (
        rel_matches_edited[col].map(has_match)
        & rel_matches_cohyponym_edited[col].map(has_match)
    )

    for idx in rel_matches_cohyponym_edited.index[mask]:
        edit_log.append({
            "row": idx,
            "model": col,
            "edited_category": "co-hyponym",
            "original_value": rel_matches_cohyponym_edited.at[idx, col],
            "new_value": "[]",
            "reason": "gold + co-hyponym match -> gold survives"
        })

    rel_matches_cohyponym_edited.loc[mask, col] = "[]"


    # ---------------------------------------------------------
    # 3. Gold + Hypernym -> Gold survives
    #    Cancel HYPERNYM
    # ---------------------------------------------------------
    mask = (
        rel_matches_edited[col].map(has_match)
        & rel_matches_hypernym_edited[col].map(has_match)
    )

    for idx in rel_matches_hypernym_edited.index[mask]:
        edit_log.append({
            "row": idx,
            "model": col,
            "edited_category": "hypernym",
            "original_value": rel_matches_hypernym_edited.at[idx, col],
            "new_value": "[]",
            "reason": "gold + hypernym match -> gold survives"
        })

    rel_matches_hypernym_edited.loc[mask, col] = "[]"


# Convert edit history to a DataFrame
edit_log_df = pd.DataFrame(edit_log)

# Optional: add 1-based row number for easier comparison with CSV files
if not edit_log_df.empty:
    edit_log_df["csv_row"] = edit_log_df["row"] + 2  # +1 for Python index, +1 for header


# Save edited files
rel_matches_edited.to_csv(
    "./wmt_json_translations/results/rel/tgt_term_gold_edited.csv",
    index=False,
    encoding="utf-8-sig"
)

rel_matches_hyponym_edited.to_csv(
    "./wmt_json_translations/results/rel/tgt_term_hyponym_edited.csv",
    index=False,
    encoding="utf-8-sig"
)

rel_matches_cohyponym_edited.to_csv(
    "./wmt_json_translations/results/rel/tgt_term_co-hyponym_edited.csv",
    index=False,
    encoding="utf-8-sig"
)

rel_matches_hypernym_edited.to_csv(
    "./wmt_json_translations/results/rel/tgt_term_hypernym_edited.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save full edit history
edit_log_df.to_csv(
    "./wmt_json_translations/results/rel/postprocessing_edit_log.csv",
    index=False,
    encoding="utf-8-sig"
)